In [1]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — ti_factors anchor sweep
# Source: TI Tables sheet → 3 tables side-by-side
#   1. Ongoing Cost basis
#   2. First Year Cost basis
#   3. Costs Spread Uniformly (4-year spread)
# Each: Age Next | Non-AIDS M | Non-AIDS F | AIDS M | AIDS F
# ─────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl import load_workbook

BASE   = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE = BASE / "Basis and Assumptions.xlsx"

wb = load_workbook(SOURCE, data_only=True, read_only=False)

# Confirm sheet name first — might be "TI Tables" or variation
print("── Available sheets ──")
for s in wb.sheetnames:
    print(f"   • {s}")

# Assume "TI Tables" per handover; adjust if diagnostic shows otherwise
TI_SHEET = "TI Tables"
assert TI_SHEET in wb.sheetnames, f"Sheet '{TI_SHEET}' not found. Available: {wb.sheetnames}"
ws = wb[TI_SHEET]

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits

def dump_block(ws, r0, c0, n_rows=12, n_cols=6, label=""):
    print(f"\n── {label}  anchor at (row={r0}, col={c0}) ──")
    print(f"{'row':>4} | " + " | ".join(f"c{c0+i:>2}" for i in range(n_cols)))
    print("-" * (7 + n_cols * 10))
    for dr in range(n_rows):
        r = r0 + dr
        vals = []
        for dc in range(n_cols):
            v = ws.cell(row=r, column=c0 + dc).value
            if v is None:
                vals.append("·")
            elif isinstance(v, float):
                vals.append(f"{v:.4g}")
            else:
                vals.append(str(v)[:8])
        print(f"{r:>4} | " + " | ".join(f"{s:>8}" for s in vals))

# ── 1) Probe title labels for the 3 cost bases ─────────────────
print("\n═══ Cost-basis title probe ═══")
title_candidates = [
    "Ongoing",              "Ongoing Cost",           "Ongoing Cost basis",
    "First Year",           "First Year Cost",        "First Year Cost basis",
    "Uniform",              "Costs Spread Uniformly", "Uniform Cost",
    "4-year spread",        "Costs Spread",
]
for label in title_candidates:
    hits = find_anchors(ws, label)
    if hits:
        print(f"   {label!r:>28} → {hits}")

# ── 2) Probe column headers ────────────────────────────────────
print("\n═══ Column header probe ═══")
col_candidates = [
    "Age Next", "Age Next Birthday", "ANB", "Age",
    "Non-AIDS M", "Non-AIDS F", "Non-Aids M", "Non-Aids F",
    "AIDS M", "AIDS F", "Aids M", "Aids F",
    "Males", "Females",
]
for label in col_candidates:
    hits = find_anchors(ws, label)
    if hits:
        print(f"   {label!r:>22} → {hits[:6]}{' …' if len(hits) > 6 else ''}  (total={len(hits)})")

# ── 3) Sheet dimensions ────────────────────────────────────────
print(f"\n═══ Sheet dimensions ═══")
print(f"   max_row = {ws.max_row}, max_col = {ws.max_column}")

# ── 4) Dump the top-left corner to see the layout ──────────────
print("\n═══ Top-left corner (rows 1–15, cols 1–20) ═══")
for r in range(1, 16):
    row_vals = []
    for c in range(1, 21):
        v = ws.cell(row=r, column=c).value
        if v is None:
            row_vals.append("·")
        elif isinstance(v, float):
            row_vals.append(f"{v:.3g}")
        else:
            row_vals.append(str(v)[:7])
    print(f"   r{r:>2}: " + " | ".join(f"{s:>7}" for s in row_vals))

── Available sheets ──
   • GLA Basis
   • LSDB Basis
   • CI Basis
   • Assumptions
   • Spouse Basis
   • TI Tables
   • ADB Basis
   • AccDB Basis
   • Dismemberment Basis

═══ Cost-basis title probe ═══
                 'Ongoing Cost' → [(3, 2)]
              'First Year Cost' → [(3, 8)]
       'Costs Spread Uniformly' → [(3, 14)]

═══ Column header probe ═══
               'Age Next' → [(4, 2), (4, 8), (4, 14)]  (total=3)

═══ Sheet dimensions ═══
   max_row = 69, max_col = 18

═══ Top-left corner (rows 1–15, cols 1–20) ═══
   r 1:       · | Termina |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · | Spread  |       · |       4 |       · |       · |       · |       ·
   r 2:       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       ·
   r 3:       · | Ongoing |       · |       · |      

In [2]:
# ═══════════════════════════════════════════════════════════════
# build_ti_factors.ipynb — Cell 1: Setup
# ═══════════════════════════════════════════════════════════════
# Builds the 'ti_factors' sheet of the Basis Master.
# Schema: cost_basis | age | non_aids_m | non_aids_f | aids_m | aids_f
# Cost bases: ongoing, first_year, uniform
# NOTE: no product column — TI is its own product.
# Expected rows: 195 (65 ages × 3 cost bases)
# ═══════════════════════════════════════════════════════════════

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Paths ──────────────────────────────────────────────────────
BASE          = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE        = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX   = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR  = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG     = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME    = "ti_factors"
PARQUET_PATH  = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"
BASIS_VERSION = "B2026_07"
BUILT_BY      = "Mohamed Hussein Gadiwala"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(),       f"Source workbook not found: {SOURCE}"
assert SNAPSHOT_DIR.exists(), f"Snapshot dir missing:      {SNAPSHOT_DIR}"

if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

print("✅ Cell 1 — Setup complete")
print(f"   Target sheet  : {SHEET_NAME}")

✅ Cell 1 — Setup complete
   Target sheet  : ti_factors


In [3]:
# ═══════════════════════════════════════════════════════════════
# build_ti_factors.ipynb — Cell 2: Helpers
# ═══════════════════════════════════════════════════════════════

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits


def as_str(v):
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def as_int(v):
    if v is None:
        return None
    if isinstance(v, int) and not isinstance(v, bool):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(s)
    except ValueError:
        try:
            f = float(s)
            return int(f) if f.is_integer() else None
        except ValueError:
            return None


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


print("✅ Cell 2 — Helpers loaded")

✅ Cell 2 — Helpers loaded


In [4]:
# ═══════════════════════════════════════════════════════════════
# build_ti_factors.ipynb — Cell 3: SPECS + header verification
# ═══════════════════════════════════════════════════════════════
# Layout per cost_basis (3 tables side-by-side on TI Tables sheet):
#   Row 3: title (Ongoing Cost / First Year Cost / Costs Spread Uniformly)
#   Row 4: Age Next | Non-AIDS M | Non-AIDS F | AIDS M | AIDS F  (assumed)
#   Rows 5..69: age | non_aids_m | non_aids_f | aids_m | aids_f
#
# Cell 3 VERIFIES the actual M/F column ordering before extraction.
# ═══════════════════════════════════════════════════════════════

SPECS = [
    {
        "cost_basis":       "ongoing",
        "title_anchor":     "Ongoing Cost",
        "age_anchor_col":   2,      # col B
    },
    {
        "cost_basis":       "first_year",
        "title_anchor":     "First Year Cost",
        "age_anchor_col":   8,      # col H
    },
    {
        "cost_basis":       "uniform",
        "title_anchor":     "Costs Spread Uniformly",
        "age_anchor_col":   14,     # col N
    },
]

TI_SHEET_NAME = "TI Tables"
HEADER_ROW    = 4
DATA_START    = 5
EXPECTED_ROWS_PER_BASIS = 65  # ages 16..80

wb = load_workbook(SOURCE, data_only=True, read_only=False)
assert TI_SHEET_NAME in wb.sheetnames, f"'{TI_SHEET_NAME}' not found"
ws = wb[TI_SHEET_NAME]

# ── Anchor validation ─────────────────────────────────────────
print("🔍 Anchor validation:")
for spec in SPECS:
    hits = find_anchors(ws, spec["title_anchor"])
    assert len(hits) == 1, f"expected 1 hit for '{spec['title_anchor']}', got {hits}"
    r, c = hits[0]
    assert c == spec["age_anchor_col"], (
        f"[{spec['cost_basis']}] title col mismatch: expected {spec['age_anchor_col']}, got {c}"
    )
    print(f"   {spec['cost_basis']:<11} → title at (row={r}, col={c})  "
          f"age col={spec['age_anchor_col']}")

# ── FULL header verification (untruncated) ─────────────────────
print("\n📋 Actual header cells (row 4, untruncated):")
print(f"   {'cost_basis':<11} | {'col':>3} | header text")
print(f"   {'-'*11} | {'-'*3} | {'-'*40}")
for spec in SPECS:
    base_col = spec["age_anchor_col"]
    for offset in range(5):
        c = base_col + offset
        raw = ws.cell(row=HEADER_ROW, column=c).value
        header = as_str(raw)
        print(f"   {spec['cost_basis']:<11} | {c:>3} | {header!r}")
    print(f"   {'-'*11} | {'-'*3} | {'-'*40}")

# ── Auto-detect M/F column offsets from headers ────────────────
# For each spec, we'll find which offset holds M vs F for both
# Non-AIDS and AIDS pairs by scanning the header text.
print("\n🎯 Auto-detecting M/F column offsets:")
COLUMN_OFFSETS = {}  # cost_basis -> dict of {age, non_aids_m, non_aids_f, aids_m, aids_f}

for spec in SPECS:
    base_col = spec["age_anchor_col"]
    headers = {}
    for offset in range(5):
        raw = ws.cell(row=HEADER_ROW, column=base_col + offset).value
        h = as_str(raw) or ""
        headers[offset] = h.lower()

    def find_offset(match_all, match_any=None):
        """Find offset whose header contains ALL of match_all and (optionally) ANY of match_any."""
        for off, h in headers.items():
            if all(m in h for m in match_all):
                if match_any is None or any(m in h for m in match_any):
                    return off
        return None

    offsets = {
        "age":        find_offset(["age"]),
        "non_aids_m": find_offset(["non", "aids"], ["m", "male"]) or find_offset(["non-aids m"]),
        "non_aids_f": find_offset(["non", "aids"], ["f", "female"]) or find_offset(["non-aids f"]),
        "aids_m":     None,
        "aids_f":     None,
    }
    # For AIDS (without "non"), scan separately
    for off, h in headers.items():
        if "aids" in h and "non" not in h:
            if any(m in h for m in ["m", "male"]) and not any(f in h for f in ["female"]):
                if offsets["aids_m"] is None:
                    offsets["aids_m"] = off
            elif any(m in h for m in ["f", "female"]):
                if offsets["aids_f"] is None:
                    offsets["aids_f"] = off

    # Sanity: if detection failed for M/F disambiguation (e.g., both headers just say
    # "Non-AIDS Factors" or truncated), fall back to positional assumption: M first, F second.
    if offsets["non_aids_m"] is None or offsets["non_aids_f"] is None or \
       offsets["non_aids_m"] == offsets["non_aids_f"]:
        print(f"   ⚠️  {spec['cost_basis']}: Non-AIDS M/F not disambiguated from headers "
              f"→ falling back to positional (M=offset 1, F=offset 2)")
        offsets["non_aids_m"] = 1
        offsets["non_aids_f"] = 2
    if offsets["aids_m"] is None or offsets["aids_f"] is None or \
       offsets["aids_m"] == offsets["aids_f"]:
        print(f"   ⚠️  {spec['cost_basis']}: AIDS M/F not disambiguated from headers "
              f"→ falling back to positional (M=offset 3, F=offset 4)")
        offsets["aids_m"] = 3
        offsets["aids_f"] = 4

    COLUMN_OFFSETS[spec["cost_basis"]] = offsets
    print(f"   {spec['cost_basis']:<11} → offsets: {offsets}")

print("\n✅ Cell 3 — SPECS loaded, headers verified")

🔍 Anchor validation:
   ongoing     → title at (row=3, col=2)  age col=2
   first_year  → title at (row=3, col=8)  age col=8
   uniform     → title at (row=3, col=14)  age col=14

📋 Actual header cells (row 4, untruncated):
   cost_basis  | col | header text
   ----------- | --- | ----------------------------------------
   ongoing     |   2 | 'Age Next'
   ongoing     |   3 | 'Non-AIDS Factor Male'
   ongoing     |   4 | 'Non-AIDS Factor Female'
   ongoing     |   5 | 'AIDS Factor Male'
   ongoing     |   6 | 'AIDS Factor Female'
   ----------- | --- | ----------------------------------------
   first_year  |   8 | 'Age Next'
   first_year  |   9 | 'Non-AIDS Factor Male'
   first_year  |  10 | 'Non-AIDS Factor Female'
   first_year  |  11 | 'AIDS Factor Male'
   first_year  |  12 | 'AIDS Factor Female'
   ----------- | --- | ----------------------------------------
   uniform     |  14 | 'Age Next'
   uniform     |  15 | 'Non-AIDS Factor Male'
   uniform     |  16 | 'Non-AIDS Factor F

In [5]:
# ═══════════════════════════════════════════════════════════════
# build_ti_factors.ipynb — Cell 4: Extract
# ═══════════════════════════════════════════════════════════════
# Walk each cost_basis table from DATA_START, break on blank age.
# ═══════════════════════════════════════════════════════════════

all_rows = []
MAX_WALK = 100  # safety cap (expected 65)

for spec in SPECS:
    cost_basis = spec["cost_basis"]
    base_col   = spec["age_anchor_col"]
    offsets    = COLUMN_OFFSETS[cost_basis]

    r = DATA_START
    rows_extracted = 0
    while r < DATA_START + MAX_WALK:
        age = as_int(ws.cell(row=r, column=base_col + offsets["age"]).value)
        if age is None:
            break

        all_rows.append({
            "cost_basis": cost_basis,
            "age":        age,
            "non_aids_m": as_float(ws.cell(row=r, column=base_col + offsets["non_aids_m"]).value),
            "non_aids_f": as_float(ws.cell(row=r, column=base_col + offsets["non_aids_f"]).value),
            "aids_m":     as_float(ws.cell(row=r, column=base_col + offsets["aids_m"]).value),
            "aids_f":     as_float(ws.cell(row=r, column=base_col + offsets["aids_f"]).value),
        })
        rows_extracted += 1
        r += 1

    print(f"   {cost_basis:<11} → {rows_extracted} rows extracted "
          f"(expected {EXPECTED_ROWS_PER_BASIS})")

df = pl.DataFrame(all_rows, infer_schema_length=None)

print(f"\n✅ Cell 4 — Extract complete")
print(f"   Total rows        : {df.height}")
print(f"   Distinct cost_basis: {df['cost_basis'].n_unique()}")
print(f"   Schema            : {df.schema}")
print("\n── Preview per cost_basis (first + last row) ──")
for cb in df["cost_basis"].unique(maintain_order=True).to_list():
    sub = df.filter(pl.col("cost_basis") == cb)
    print(f"\n{cb} ({sub.height} rows):")
    print(sub.head(2))
    print(sub.tail(2))

   ongoing     → 65 rows extracted (expected 65)
   first_year  → 65 rows extracted (expected 65)
   uniform     → 65 rows extracted (expected 65)

✅ Cell 4 — Extract complete
   Total rows        : 195
   Distinct cost_basis: 3
   Schema            : Schema([('cost_basis', String), ('age', Int64), ('non_aids_m', Float64), ('non_aids_f', Float64), ('aids_m', Float64), ('aids_f', Float64)])

── Preview per cost_basis (first + last row) ──

ongoing (65 rows):
shape: (2, 6)
┌────────────┬─────┬────────────┬────────────┬──────────┬──────────┐
│ cost_basis ┆ age ┆ non_aids_m ┆ non_aids_f ┆ aids_m   ┆ aids_f   │
│ ---        ┆ --- ┆ ---        ┆ ---        ┆ ---      ┆ ---      │
│ str        ┆ i64 ┆ f64        ┆ f64        ┆ f64      ┆ f64      │
╞════════════╪═════╪════════════╪════════════╪══════════╪══════════╡
│ ongoing    ┆ 16  ┆ 0.0015     ┆ 0.003      ┆ 0.0      ┆ 0.392874 │
│ ongoing    ┆ 17  ┆ 0.0007     ┆ 0.002      ┆ 0.003963 ┆ 0.295629 │
└────────────┴─────┴────────────┴────────

In [6]:
# ═══════════════════════════════════════════════════════════════
# PROBE — dump exact source values for SPOT_CHECKS
# Reads the raw cell values (full float64 precision) from the
# source workbook and prints a ready-to-paste SPOT_CHECKS list.
# ═══════════════════════════════════════════════════════════════

# Which (cost_basis, age, col) tuples we want to spot-check
PROBE_TARGETS = [
    ("ongoing",    16, "non_aids_m"),
    ("ongoing",    16, "non_aids_f"),
    ("ongoing",    16, "aids_m"),
    ("ongoing",    16, "aids_f"),
    ("ongoing",    18, "aids_m"),
    ("first_year", 16, "non_aids_m"),
    ("first_year", 16, "non_aids_f"),
    ("first_year", 16, "aids_m"),
    ("first_year", 16, "aids_f"),
    ("uniform",    16, "non_aids_m"),
    ("uniform",    16, "non_aids_f"),
    ("uniform",    16, "aids_m"),
    ("uniform",    16, "aids_f"),
]

# Base col per cost_basis (from Cell 3)
BASE_COL = {"ongoing": 2, "first_year": 8, "uniform": 14}

# Offset per column (positional — confirmed from headers row 4)
COL_OFFSET = {
    "non_aids_m": 1,
    "non_aids_f": 2,
    "aids_m":     3,
    "aids_f":     4,
}

# DATA_START row 5 corresponds to age 16, so age N is at row 5 + (N - 16)
def row_for_age(age):
    return 5 + (age - 16)

print("SPOT_CHECKS = [")
for cost_basis, age, col in PROBE_TARGETS:
    r = row_for_age(age)
    c = BASE_COL[cost_basis] + COL_OFFSET[col]
    raw = ws.cell(row=r, column=c).value
    # repr gives full float64 precision
    print(f"    ({cost_basis!r:<13}, {age:>2}, {col!r:<12}, {raw!r}),")
print("]")

SPOT_CHECKS = [
    ('ongoing'    , 16, 'non_aids_m', 0.0015),
    ('ongoing'    , 16, 'non_aids_f', 0.003),
    ('ongoing'    , 16, 'aids_m'    , 0),
    ('ongoing'    , 16, 'aids_f'    , 0.39287426022435956),
    ('ongoing'    , 18, 'aids_m'    , 0.4696595182810219),
    ('first_year' , 16, 'non_aids_m', 0.0093),
    ('first_year' , 16, 'non_aids_f', 0.0102),
    ('first_year' , 16, 'aids_m'    , 0.05825200037638339),
    ('first_year' , 16, 'aids_f'    , 0.5053742602243595),
    ('uniform'    , 16, 'non_aids_m', 0.00345),
    ('uniform'    , 16, 'non_aids_f', 0.0048000000000000004),
    ('uniform'    , 16, 'aids_m'    , 0.014563000094095847),
    ('uniform'    , 16, 'aids_f'    , 0.4209992602243595),
]


In [7]:
# ═══════════════════════════════════════════════════════════════
# build_ti_factors.ipynb — Cell 6: Write outputs + build log
# ═══════════════════════════════════════════════════════════════

# Parquet
df.write_parquet(PARQUET_PATH)
parquet_hash = sha256_of(PARQUET_PATH)
parquet_kb   = round(PARQUET_PATH.stat().st_size / 1024, 2)
print(f"📦 Parquet written : {PARQUET_PATH.name}  ({parquet_kb} KB)")

# Master xlsx guard
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

if MASTER_XLSX.exists():
    wb_master = load_workbook(MASTER_XLSX)
else:
    from openpyxl import Workbook
    wb_master = Workbook()
    if "Sheet" in wb_master.sheetnames and len(wb_master.sheetnames) == 1:
        del wb_master["Sheet"]

if SHEET_NAME in wb_master.sheetnames:
    del wb_master[SHEET_NAME]

ws_new = wb_master.create_sheet(SHEET_NAME)
columns = df.columns
ws_new.append(columns)
for row in df.iter_rows():
    ws_new.append(list(row))

header_row = next(ws_new.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for col_idx, col_name in enumerate(columns, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(v)) if v is not None else 0 for v in df[col_name].to_list()]
    )
    ws_new.column_dimensions[get_column_letter(col_idx)].width = max(12, max_len + 2)

last_col_letter = get_column_letter(len(columns))
table_ref = f"A1:{last_col_letter}{df.height + 1}"
tbl = Table(displayName=f"tbl_{SHEET_NAME}", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_new.add_table(tbl)

wb_master.save(MASTER_XLSX)
print(f"📗 Master xlsx     : sheet '{SHEET_NAME}' upserted")

# Build log
source_hash = sha256_of(SOURCE)

log_row = {
    "timestamp_utc":     utc_now_iso(),
    "sheet":             SHEET_NAME,
    "row_count":         df.height,
    "distinct_products": df["cost_basis"].n_unique(),  # cost_basis serves as product-analog
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      PARQUET_PATH.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        parquet_kb,
    "status":            "ok",
    "built_by":          BUILT_BY,
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log       : appended row for '{SHEET_NAME}'")
print(f"\n✅ Cell 6 — ti_factors build COMPLETE")
print(f"   Rows        : {df.height}")
print(f"   Cost bases  : {sorted(df['cost_basis'].unique().to_list())}")

📦 Parquet written : ti_factors.parquet  (5.69 KB)
📗 Master xlsx     : sheet 'ti_factors' upserted
📝 Build log       : appended row for 'ti_factors'

✅ Cell 6 — ti_factors build COMPLETE
   Rows        : 195
   Cost bases  : ['first_year', 'ongoing', 'uniform']
